# Environment and credentials check

Verifies that the external services the knowledge-base pipeline depends on are reachable and
correctly configured: Weaviate Cloud, the Cohere embedding API, and the LLM API. All checks
are read-only. Credential values are never printed; cells report only whether a variable is
set and its length.

## 1. Load environment

Finds the nearest `.env` file, searching upward from the working directory, and loads it into
the process environment. Prints the path that was used.

In [ ]:
import json
import os
import urllib.error
import urllib.request
from urllib.parse import urlparse

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
print(f"loaded .env from: {env_path or '<not found>'}  ({load_dotenv(env_path)})")

## 2. Static validation

Reads the Weaviate and embedding variables and checks them without any network call: required
keys are non-empty and `WEAVIATE_URL` is a well-formed `https://` URL. Defines the helper
functions used by later cells to display credentials as presence and length only.

In [ ]:
def shown(v: str) -> str:
    return f"set ({len(v.strip())} chars)" if v and v.strip() else "MISSING"


def redact_url(u: str) -> str:
    if not u:
        return "MISSING"
    p = urlparse(u)
    return f"{p.scheme}://….{'.'.join((p.hostname or '').split('.')[-2:])}"


WEAVIATE_URL = os.environ.get("WEAVIATE_URL", "")
WEAVIATE_API_KEY = os.environ.get("WEAVIATE_API_KEY", "")
WEAVIATE_READ_API_KEY = os.environ.get("WEAVIATE_READ_API_KEY", "")
EMBEDDING_PROVIDER = os.environ.get("EMBEDDING_PROVIDER", "cohere").lower()
EMBEDDING_API_KEY = os.environ.get("EMBEDDING_API_KEY", "")

print(f"WEAVIATE_URL          = {redact_url(WEAVIATE_URL)}")
print(f"WEAVIATE_API_KEY      = {shown(WEAVIATE_API_KEY)}")
print(f"WEAVIATE_READ_API_KEY = {shown(WEAVIATE_READ_API_KEY)}")
print(f"EMBEDDING_PROVIDER    = {EMBEDDING_PROVIDER}")
print(f"EMBEDDING_API_KEY     = {shown(EMBEDDING_API_KEY)}")

problems: list[str] = []
if not WEAVIATE_URL or "your-cluster" in WEAVIATE_URL:
    problems.append("WEAVIATE_URL missing or still the placeholder")
elif urlparse(WEAVIATE_URL).scheme != "https" or not urlparse(WEAVIATE_URL).netloc:
    problems.append("WEAVIATE_URL malformed (not https:// or no host)")
if not WEAVIATE_API_KEY:
    problems.append("WEAVIATE_API_KEY empty")
if not EMBEDDING_API_KEY:
    problems.append(f"EMBEDDING_API_KEY empty (needed for provider {EMBEDDING_PROVIDER})")

print()
print("static check:", ("FAIL -> " + "; ".join(problems)) if problems else "ok")

## 3. Weaviate Cloud

Connects to the cluster with each API key and confirms authentication. The embedding key is
sent as the vectorizer header but is not exercised here.

### 3.1 Admin key

Connects with the read-write key used by the load and delete scripts. Prints readiness,
server version, the list of collections, and the object count of the `KnowledgeBase`
collection when it exists (197 for the full corpus).

In [ ]:
import weaviate
from weaviate.classes.init import Auth

EMB_HEADER = "X-OpenAI-Api-Key" if EMBEDDING_PROVIDER == "openai" else "X-Cohere-Api-Key"

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
    headers={EMB_HEADER: EMBEDDING_API_KEY or "unset"},
)
try:
    print("is_ready()  =", client.is_ready())
    print("server      =", client.get_meta().get("version"))
    collections = list(client.collections.list_all())
    print("collections =", collections or "(none)")
    if "KnowledgeBase" in collections:
        kb = client.collections.get("KnowledgeBase")
        total = kb.aggregate.over_all(total_count=True).total_count
        print(f"KnowledgeBase count = {total}  (expect 197)")
    else:
        print("KnowledgeBase not created yet -> run scripts/load_knowledge_base.py")
finally:
    client.close()

### 3.2 Read-only key

Connects with the key intended for the retrieval path. Confirms it authenticates and can list
collections. A missing value is reported rather than treated as a failure.

In [ ]:
if not WEAVIATE_READ_API_KEY:
    print("WEAVIATE_READ_API_KEY MISSING (unused in phase 1)")
else:
    c = weaviate.connect_to_weaviate_cloud(
        cluster_url=WEAVIATE_URL,
        auth_credentials=Auth.api_key(WEAVIATE_READ_API_KEY),
        headers={EMB_HEADER: EMBEDDING_API_KEY or "unset"},
    )
    try:
        print("is_ready() =", c.is_ready())
        print("collections =", list(c.collections.list_all()) or "(none)")
        print("-> read key authenticates OK")
    finally:
        c.close()

## 4. Cohere embeddings (`EMBEDDING_API_KEY`)

A misconfigured embedding key is not reported by Weaviate until the first vectorized
operation, so the Cohere embedding endpoint is called directly.

### 4.1 Key validity

Sends `Hello!` to the Cohere embedding endpoint and checks the response status and the
returned vector dimension.

In [ ]:
def cohere_embed(texts: list[str], input_type: str) -> list[list[float]]:
    body = json.dumps(
        {
            "model": "embed-english-v3.0",
            "input_type": input_type,
            "embedding_types": ["float"],
            "texts": texts,
        }
    ).encode()
    req = urllib.request.Request(
        "https://api.cohere.com/v2/embed",
        data=body,
        headers={
            "Authorization": f"Bearer {EMBEDDING_API_KEY.strip()}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.load(resp)["embeddings"]["float"]


if EMBEDDING_PROVIDER != "cohere":
    print(f"EMBEDDING_PROVIDER={EMBEDDING_PROVIDER}; skipping Cohere check")
elif not EMBEDDING_API_KEY:
    print("EMBEDDING_API_KEY MISSING")
else:
    try:
        vec = cohere_embed(["Hello!"], "search_query")[0]
        print(f"HTTP 200 -- {len(vec)}-dim embedding -> Cohere key VALID")
    except urllib.error.HTTPError as e:
        print(f"HTTP {e.code}: {e.read().decode()[:300]}")
        print("-> Cohere key INVALID or lacks permission")

### 4.2 Semantic similarity example

Embeds a sample query and two candidate documents in the same form the corpus stores, then
prints the cosine similarity of each. The query and documents use the distinct
`search_query` and `search_document` input types that the retrieval path uses; the topically
related document is expected to score higher.

In [ ]:
import math


def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b, strict=True))
    norm = math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b))
    return dot / norm


if EMBEDDING_PROVIDER == "cohere" and EMBEDDING_API_KEY:
    query = "something spicy with noodles"
    docs = [
        "Firecracker chicken ramen. Category: Ramen. "
        "Wok-fried chilli chicken in a fiery broth with noodles.",
        "Vanilla ice cream. Category: Desserts. Three scoops of dairy vanilla ice cream.",
    ]
    qv = cohere_embed([query], "search_query")[0]
    dvs = cohere_embed(docs, "search_document")
    scores = [cosine(qv, dv) for dv in dvs]
    print(f"query: {query!r}")
    for d, s in zip(docs, scores, strict=True):
        print(f"  cos={s:+.3f}  {d[:60]}...")
    verdict = "ramen (sensible)" if scores[0] > scores[1] else "dessert (UNEXPECTED)"
    print(f"\n-> closest line is the {verdict}")
else:
    print("skipped (Cohere key not available)")

## 5. LLM (`LLM_API_KEY`)

Checks the language-model credentials. Reads the provider, model name, and API key from the
environment.

### 5.1 Live call example

Sends a single `Hello!` message to the provider's REST endpoint and prints the response
text. The model endpoint may take 10-30 seconds to respond. Runs only when an API key is
set.

In [ ]:
LLM_PROVIDER = os.environ.get("LLM_PROVIDER", "google").lower()
LLM_MODEL = os.environ.get("LLM_MODEL", "gemini-3.8-flash")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "")

print(f"LLM_PROVIDER = {LLM_PROVIDER}")
print(f"LLM_MODEL    = {LLM_MODEL}")
print(f"LLM_API_KEY  = {shown(LLM_API_KEY)}")


def gemini_say(prompt: str) -> str:
    url = (
        f"https://generativelanguage.googleapis.com/v1beta/models/{LLM_MODEL}:generateContent"
        f"?key={LLM_API_KEY.strip()}"
    )
    body = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode()
    req = urllib.request.Request(
        url, data=body, headers={"Content-Type": "application/json"}, method="POST"
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.load(resp)
    parts = data["candidates"][0]["content"]["parts"]
    return " ".join(p.get("text", "") for p in parts).strip()


if not LLM_API_KEY:
    print("\nLLM_API_KEY not set -- skipping live call. Fill it in .env to test (phase 2).")
elif LLM_PROVIDER == "google":
    try:
        answer = gemini_say("Hello!")
        print("\nprompt   : Hello!")
        print(f"response : {answer}")
        print("\n-> LLM key VALID" if answer else "\n-> empty response")
    except urllib.error.HTTPError as e:
        print(f"\nHTTP {e.code}: {e.read().decode()[:300]}")
        print("-> LLM key INVALID or lacks permission")
    except (TimeoutError, urllib.error.URLError) as e:
        print(f"\nrequest failed: {e}")
        print("-> could not reach the LLM endpoint (network, or endpoint too slow)")
else:
    print(f"\nprovider {LLM_PROVIDER!r}: no client wired into this check")

## 6. Summary

Once every check above reports success, the environment is ready to run
`scripts/load_knowledge_base.py`.